# Tutorial: KNN in Real Life - Netflix-like Recommendations

Audience: Beginners in ML and recommendation systems.

Prerequisites: Basic Python, lists/dictionaries, and the idea of vectors.

Learning goals:
- Understand how KNN can power movie recommendations.
- Build a tiny similarity-based recommender from scratch.


## Outline
1. Real-life intuition (Netflix-like behavior)
2. Build a tiny user-rating dataset
3. Compute user-to-user similarity
4. Find nearest neighbors
5. Recommend unseen movies
6. Exercise and answer scaffold


## 1) Real-life intuition
KNN (K-Nearest Neighbors) for recommendations works like this:
- Represent each user by their movie-rating pattern.
- Find users with similar taste (nearest neighbors).
- Recommend movies those neighbors liked but the target user has not watched.

This is a simple form of collaborative filtering.


In [1]:
# 2) Tiny dataset: ratings from 0 (not watched) to 5
users = {
    "Ava":   {"Inception": 5, "Interstellar": 5, "Titanic": 1, "Notebook": 1, "Avengers": 4},
    "Ben":   {"Inception": 4, "Interstellar": 5, "Titanic": 1, "Notebook": 1, "Avengers": 5},
    "Cara":  {"Inception": 1, "Interstellar": 1, "Titanic": 5, "Notebook": 4, "Avengers": 1},
    "Dan":   {"Inception": 2, "Interstellar": 2, "Titanic": 5, "Notebook": 5, "Avengers": 2},
    "Eli":   {"Inception": 5, "Interstellar": 4, "Titanic": 0, "Notebook": 0, "Avengers": 4},
}

target_user = "Eli"
movies = list(next(iter(users.values())).keys())

print("Target user:", target_user)
print("Movies:", movies)
print("Ratings for Eli:", users[target_user])


Target user: Eli
Movies: ['Inception', 'Interstellar', 'Titanic', 'Notebook', 'Avengers']
Ratings for Eli: {'Inception': 5, 'Interstellar': 4, 'Titanic': 0, 'Notebook': 0, 'Avengers': 4}


In [2]:
# 3) Similarity function (cosine similarity)
import math

def to_vector(user_ratings, movie_order):
    return [user_ratings[m] for m in movie_order]

def cosine_similarity(v1, v2):
    dot = sum(a*b for a, b in zip(v1, v2))
    n1 = math.sqrt(sum(a*a for a in v1))
    n2 = math.sqrt(sum(b*b for b in v2))
    return 0 if n1 == 0 or n2 == 0 else dot / (n1 * n2)

target_vec = to_vector(users[target_user], movies)
similarities = {}
for other_user, ratings in users.items():
    if other_user == target_user:
        continue
    other_vec = to_vector(ratings, movies)
    similarities[other_user] = cosine_similarity(target_vec, other_vec)

for name, score in sorted(similarities.items(), key=lambda x: x[1], reverse=True):
    print(f"Similarity(Eli, {name}) = {score:.3f}")


Similarity(Eli, Ava) = 0.980
Similarity(Eli, Ben) = 0.964
Similarity(Eli, Dan) = 0.437
Similarity(Eli, Cara) = 0.260


In [3]:
# 4) Pick K nearest neighbors
K = 2
neighbors = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:K]
print("Top neighbors:")
for name, score in neighbors:
    print(f"- {name}: {score:.3f}")


Top neighbors:
- Ava: 0.980
- Ben: 0.964


In [4]:
# 5) Recommend movies Eli has not watched (rating == 0)
def recommend_for_user(users, target_user, neighbors, movie_order):
    rec_scores = {}
    target_ratings = users[target_user]

    for movie in movie_order:
        if target_ratings[movie] != 0:
            continue  # already watched

        weighted_sum = 0.0
        sim_sum = 0.0
        for neighbor_name, sim in neighbors:
            rating = users[neighbor_name][movie]
            if rating > 0:
                weighted_sum += sim * rating
                sim_sum += sim

        rec_scores[movie] = 0 if sim_sum == 0 else weighted_sum / sim_sum

    return sorted(rec_scores.items(), key=lambda x: x[1], reverse=True)

recommendations = recommend_for_user(users, target_user, neighbors, movies)
print("Recommended for Eli:")
for movie, score in recommendations:
    print(f"- {movie}: predicted interest {score:.2f}/5")


Recommended for Eli:
- Titanic: predicted interest 1.00/5
- Notebook: predicted interest 1.00/5


## 6) Exercise
Change `K` from `2` to `3` and rerun neighbor selection + recommendation.
- Which movie remains top recommendation?
- Do predicted scores change significantly?


In [5]:
# Answer scaffold
K = 3
neighbors_k3 = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:K]
recommendations_k3 = recommend_for_user(users, target_user, neighbors_k3, movies)

print("Top neighbors for K=3:")
for name, score in neighbors_k3:
    print(f"- {name}: {score:.3f}")

print("\nRecommendations for K=3:")
for movie, score in recommendations_k3:
    print(f"- {movie}: predicted interest {score:.2f}/5")


Top neighbors for K=3:
- Ava: 0.980
- Ben: 0.964
- Dan: 0.437

Recommendations for K=3:
- Titanic: predicted interest 1.73/5
- Notebook: predicted interest 1.73/5


## Pitfall and extension
Common pitfall: Treating missing ratings as real zeros can distort similarity.
Fix: In real systems, compute similarity only on co-rated items.

Extension: Try item-based KNN (find similar movies instead of similar users).
